# Kimi K3 From Scratch

Minimal pure-PyTorch walkthrough of the **Kimi K3** text architecture.

| Component | What it does |
| --- | --- |
| **KDA** | Linear attention with channel-wise delta-rule state (~¾ of layers) |
| **Gated MLA** | Full latent attention + output gate (~¼ of layers) |
| **LatentMoE** | Sparse experts in a compressed latent space |
| **SiTU-GLU** | Soft-capped tanh × sigmoid (not SwiGLU) |
| **Block AttnRes** | Softmax residual mix over layer history |

Default size matches [`inference-optimization/Kimi-K3-0.18B`](https://huggingface.co/inference-optimization/Kimi-K3-0.18B)  
(full model: [`moonshotai/Kimi-K3`](https://huggingface.co/moonshotai/Kimi-K3)).

Implementation lives in `.py` modules; this notebook is the structured entry point.


## Setup

Run from the repository root.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "kimi_k3.py").exists():
    raise FileNotFoundError("Open this notebook from the repository root")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
from kimi_k3 import (
    KIMI_K3_CONFIG_0_18B,
    KIMI_K3_CONFIG_MICRO,
    KIMI_K3_FULL_REFERENCE,
    KimiK3Model,
    count_parameters,
    describe_architecture,
    layer_type,
)
from generate import generate_text_basic

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
print("torch:", torch.__version__)


## Full Kimi K3 (reference)

Scale numbers only — do **not** instantiate this dict as a model.


In [2]:
import json
print(json.dumps(KIMI_K3_FULL_REFERENCE, indent=2))


{
  "vocab_size": 160000,
  "context_length": 1048576,
  "emb_dim": 7168,
  "n_layers": 93,
  "n_heads": 96,
  "n_kv_heads": 96,
  "intermediate_size": 33792,
  "moe_intermediate_size": 3072,
  "routed_expert_hidden_size": 3584,
  "num_experts": 896,
  "num_experts_per_token": 16,
  "num_shared_experts": 2,
  "first_k_dense_replace": 1,
  "kv_lora_rank": 512,
  "q_lora_rank": 1536,
  "qk_nope_head_dim": 128,
  "qk_rope_head_dim": 64,
  "v_head_dim": 128,
  "linear_num_heads": 96,
  "linear_head_dim": 128,
  "linear_conv_kernel_size": 4,
  "attn_res_block_size": 12,
  "n_kda_layers": 69,
  "n_mla_layers": 24,
  "activation": "situ",
  "total_params": "2.8T",
  "activated_params": "104B"
}


## Tiny isomorphic config (0.18B)

Same layer types as full K3; width, depth, and expert count are reduced for local runs.


In [3]:
print("Full model (reference)")
r = KIMI_K3_FULL_REFERENCE
print(f"  layers={r['n_layers']}  KDA={r['n_kda_layers']}  MLA={r['n_mla_layers']}")
print(f"  experts={r['num_experts']}  top-k={r['num_experts_per_token']}  latent={r['routed_expert_hidden_size']}")
print(f"  total≈{r['total_params']}  active≈{r['activated_params']}")

print()
print("Tiny 0.18B-shaped config")
print(describe_architecture(KIMI_K3_CONFIG_0_18B))


Full model (reference)
  layers=93  KDA=69  MLA=24
  experts=896  top-k=16  latent=3584
  total≈2.8T  active≈104B

Tiny 0.18B-shaped config
layers=4  emb=512  vocab=163840
KDA layers (1-based): [1, 2, 3]
MLA layers (1-based): [4]
first_k_dense_replace=1
MoE: experts=8 topk=2 shared=1 latent=256
MLA ranks: q_lora=128 kv_lora=64 nope/rope/v=64/32/64
KDA: heads=4 head_dim=64 conv=4
AttnRes block size=4  act=situ
  L0: attn=kda   ffn=dense SiTU-GLU
  L1: attn=kda   ffn=LatentMoE
  L2: attn=kda   ffn=LatentMoE
  L3: attn=full  ffn=LatentMoE


### Hyperparameter checks vs public 0.18B card

In [4]:
expected = {
    "n_layers": 4,
    "emb_dim": 512,
    "n_heads": 4,
    "intermediate_size": 1024,
    "moe_intermediate_size": 256,
    "routed_expert_hidden_size": 256,
    "num_experts": 8,
    "num_experts_per_token": 2,
    "num_shared_experts": 1,
    "kv_lora_rank": 64,
    "q_lora_rank": 128,
    "qk_nope_head_dim": 64,
    "qk_rope_head_dim": 32,
    "v_head_dim": 64,
    "linear_head_dim": 64,
    "first_k_dense_replace": 1,
    "hidden_act": "situ",
    "mla_use_output_gate": True,
}
cfg = KIMI_K3_CONFIG_0_18B
for key, value in expected.items():
    status = "ok" if cfg[key] == value else "DIFF"
    print(f"{status:4s}  {key}={cfg[key]!r}")

schedule = [layer_type(cfg, i) for i in range(cfg["n_layers"])]
print("schedule:", schedule)
assert schedule == ["kda", "kda", "kda", "full"]
print("3 KDA + 1 Gated MLA")


ok    n_layers=4
ok    emb_dim=512
ok    n_heads=4
ok    intermediate_size=1024
ok    moe_intermediate_size=256
ok    routed_expert_hidden_size=256
ok    num_experts=8
ok    num_experts_per_token=2
ok    num_shared_experts=1
ok    kv_lora_rank=64
ok    q_lora_rank=128
ok    qk_nope_head_dim=64
ok    qk_rope_head_dim=32
ok    v_head_dim=64
ok    linear_head_dim=64
ok    first_k_dense_replace=1
ok    hidden_act='situ'
ok    mla_use_output_gate=True
schedule: ['kda', 'kda', 'kda', 'full']
3 KDA + 1 Gated MLA


## Where each idea lives in code

| Idea | Module | Notes |
| --- | --- | --- |
| SiTU | `kimi_k3_ops.situ` | `soft_cap(x) * sigmoid(β x)` |
| KDA recurrence | `kimi_k3_ops.recurrent_kda` | Channel-wise α, delta write |
| KDA layer | `kimi_k3.KimiDeltaAttention` | Short conv, f_a/f_b, gated o_norm |
| Gated MLA | `kimi_k3.GatedMLA` | q/kv LoRA, NOPE+RoPE, `g_proj` |
| LatentMoE | `kimi_k3.LatentMoE` | down → latent norm → experts → up |
| Block AttnRes | `kimi_k3.BlockAttnRes` | Softmax over residual bank |
| Full stack | `kimi_k3.KimiK3Model` | Embed → blocks → LM head |

The next cells use **MICRO**: identical topology, smaller dimensions (CPU-friendly).


In [5]:
torch.manual_seed(0)
micro = KIMI_K3_CONFIG_MICRO
model = KimiK3Model(micro).to(device).eval()

print(describe_architecture(micro))
n = count_parameters(model)
print(f"parameters: {n:,} ({n / 1e6:.3f} M)")

for i, block in enumerate(model.blocks):
    print(
        f"  L{i}: {type(block.self_attn).__name__} + {type(block.mlp).__name__}"
    )


layers=4  emb=64  vocab=256
KDA layers (1-based): [1, 2, 3]
MLA layers (1-based): [4]
first_k_dense_replace=1
MoE: experts=4 topk=2 shared=1 latent=32
MLA ranks: q_lora=32 kv_lora=16 nope/rope/v=16/8/16
KDA: heads=2 head_dim=16 conv=4
AttnRes block size=4  act=situ
  L0: attn=kda   ffn=dense SiTU-GLU
  L1: attn=kda   ffn=LatentMoE
  L2: attn=kda   ffn=LatentMoE
  L3: attn=full  ffn=LatentMoE
parameters: 211,218 (0.211 M)
  L0: KimiDeltaAttention + DenseFFN
  L1: KimiDeltaAttention + LatentMoE
  L2: KimiDeltaAttention + LatentMoE
  L3: GatedMLA + LatentMoE


## Forward pass

In [6]:
ids = torch.randint(0, micro["vocab_size"], (2, 16), device=device)
with torch.no_grad():
    logits, _ = model(ids)

print("input :", tuple(ids.shape))
print("logits:", tuple(logits.shape))
print("finite:", bool(torch.isfinite(logits).all()))


input : (2, 16)
logits: (2, 16, 256)
finite: True


## Generation pipeline

Weights are random: token ids are meaningless; this only checks decode wiring.


In [7]:
prompt = torch.randint(0, micro["vocab_size"], (1, 6), device=device)
out = generate_text_basic(model, prompt, max_new_tokens=8, temperature=0.0)
print("prompt:", prompt.tolist())
print("output:", out.tolist())
print("new tokens:", out.shape[1] - prompt.shape[1])


prompt: [[199, 176, 38, 43, 188, 188]]
output: [[199, 176, 38, 43, 188, 188, 250, 10, 226, 141, 194, 22, 174, 63]]
new tokens: 8


## Optional: 0.18B-shaped graph

Vocab alone makes embedding + LM head large (~168M). Enable only if memory allows.


In [8]:
RUN_0_18B = False

if RUN_0_18B:
    large = KimiK3Model(KIMI_K3_CONFIG_0_18B).to(device)
    n_large = count_parameters(large)
    print(f"params: {n_large:,} ({n_large / 1e9:.3f} B)")
    with torch.no_grad():
        x = torch.randint(0, 1000, (1, 4), device=device)
        y, _ = large(x)
    print("logits:", tuple(y.shape))
else:
    print("skipped — set RUN_0_18B = True to build the 0.18B-shaped model")


skipped — set RUN_0_18B = True to build the 0.18B-shaped model


## Optional: official tiny checkpoint

```python
from load_weights import load_hf_reference_model

hf_model, tokenizer = load_hf_reference_model(
    "inference-optimization/Kimi-K3-0.18B",
    device=device,
)
```

Downloads ~0.69 GB. Requires `transformers` and `trust_remote_code=True`.


## Architecture checklist

Confirm the stack is Kimi-K3-shaped:

1. Hybrid attention: **KDA** and **Gated MLA** at 3∶1  
2. KDA forget gate is **channel-wise** \(\alpha \in \mathbb{R}^{H \times D_k}\)  
3. Full layers use **MLA** (latent ranks), not plain MHA/GQA  
4. FFN: one **dense** layer, then **LatentMoE**  
5. Activation is **SiTU-GLU**, not SwiGLU  
6. Residuals use **Block AttnRes** over a history bank  

Source: `kimi_k3.py`, `kimi_k3_ops.py`. Overview: `README.md`.
